# Singapore HDB Resale Price Prediction

This notebook approaches the dataset from a **supervised machine-learning** perspective.

## Main goal
Predict `resale_price` from property, lease, timing, and location-related features.

## Learning roadmap
The notebook is organized as a modeling workflow that a user can follow step by step:

1. load and inspect the data
2. clean the raw fields
3. engineer predictive features
4. split the data into train / validation / test sets
5. build comparable preprocessing pipelines
6. train several model families
7. compare model performance
8. visualize predictions and interpret the best models
9. project the data into lower dimensions for intuition-building

## Why use multiple models?
Different algorithms capture different patterns:

- linear models show the simplest baseline relationship
- tree-based models capture non-linear interactions
- neural networks provide a flexible function approximator
- distance-based methods show what happens when we predict from similar past transactions

Comparing them helps the notebook teach not just **what works best**, but also **why different models behave differently**.


In [ ]:

# Optional installs (uncomment only if needed in your environment)
# %pip install tqdm umap-learn shap xgboost lightgbm


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import itertools
import math
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.colors import LogNorm

from tqdm.auto import tqdm
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import shap
import umap

from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import contextily as cx
import geopandas as gpd
from shapely.geometry import Point
from matplotlib.colors import Normalize, TwoSlopeNorm

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True

## 1. Import data

This section loads the raw resale transaction file and the geocoding results produced by the trends notebook.

### Why reuse the geocodes?
The prediction notebook is not only about property attributes like floor area or flat type.  
Location matters a lot in housing prices, so geocoded addresses are reused as predictive features.

This is also why the recommended run order is:
1. `singapore_hdb_resale_trends.ipynb`
2. `singapore_hdb_resale_prediction.ipynb`


### Import CSV data

The raw HDB CSV is the main transaction table.

The helper function below cleans only the foundational fields first:
- numeric conversion for price and floor area
- standardized address creation
- removal of clearly invalid rows

The idea is to produce a dependable starting table before feature engineering begins.


In [ ]:
# Update this path if your CSV is saved somewhere else.
DATA_PATH = Path("ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv")
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

### Function rationale: foundational cleaning before modeling

The next function does not try to create every feature at once.  
Instead, it performs only the essential cleaning needed to make the raw transaction table usable.

This separation is deliberate:
- first make the base table trustworthy
- then engineer additional features in a later step

That makes the logic easier for a learner to follow.


In [ ]:
def preprocess_hdb_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["floor_area_sqm"] = pd.to_numeric(df["floor_area_sqm"], errors="coerce")
    df["resale_price"] = pd.to_numeric(df["resale_price"], errors="coerce")

    df["block"] = df["block"].astype(str).str.strip()
    df["street_name"] = df["street_name"].astype(str).str.strip()
    df["address"] = (
        df["block"] + " " + df["street_name"]
    ).str.replace(r"\s+", " ", regex=True).str.strip()

    df = df.dropna(subset=["floor_area_sqm", "resale_price", "address"])
    df = df[df["floor_area_sqm"] > 0].copy()
    df = df.sort_values(["town", "street_name", "block"]).reset_index(drop=True)
    return df
    """Clean essential raw fields and create a normalized address key before feature engineering begins."""
    
df = preprocess_hdb_data(df)


### Import geocode data

The next cell loads `cache/address_geocodes.csv`, which was generated by the trends notebook.

### Why bring geocodes into a prediction notebook?
Address-level coordinates let us encode location more precisely than town names alone.  
Even when latitude and longitude are not perfect measures of accessibility or desirability, they often carry useful predictive signal.


In [ ]:
geocode_path = Path("cache/address_geocodes.csv")
geocodes = pd.read_csv(geocode_path)
geocodes["address"] = geocodes["address"].astype(str).str.strip()
for col in ["latitude", "longitude"]:
    geocodes[col] = pd.to_numeric(geocodes[col], errors="coerce")

df = df.merge(
    geocodes[["address", "latitude", "longitude"]].drop_duplicates("address"),
    on="address",
    how="left",
)

df[["town", "address", "floor_area_sqm", "resale_price", "latitude", "longitude"]].head()

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

In [ ]:
# Numeric summary
display(df.describe(include=[np.number]).T)

# Categorical summary
categorical_cols_raw = df.select_dtypes(include="object").columns.tolist()
display(df[categorical_cols_raw].describe().T)

In [ ]:
target_summary = pd.Series({
    "rows": len(df),
    "min_price": df["resale_price"].min(),
    "median_price": df["resale_price"].median(),
    "mean_price": df["resale_price"].mean(),
    "max_price": df["resale_price"].max(),
    "std_price": df["resale_price"].std(),
})
display(target_summary)

plt.figure(figsize=(5, 4))
plt.hist(df["resale_price"], bins=60)
plt.title("Distribution of Resale Price")
plt.xlabel("Resale Price")
plt.ylabel("Count")
plt.show()


## 2. Feature engineering

Raw columns rarely capture all the structure a model needs.

### Purpose of feature engineering here
Transform the original resale table into a richer modeling table by creating features that better reflect housing economics, including:

- time of sale
- remaining lease / property age
- floor level
- unit size
- approximate location
- price per square meter for exploratory analysis

### Why this matters
Machine-learning models can only learn from the inputs they are given.  
Feature engineering is therefore one of the most important ways to inject domain logic into a predictive workflow.


### Function rationale: translating raw text fields into numeric features

Several useful inputs are buried inside text columns such as:

- `remaining_lease`
- `storey_range`

Models work better with numeric features, so the next functions parse those text fields into quantities such as:
- remaining lease in months
- mid-point floor level

This is a classic example of turning human-readable metadata into model-readable variables.


In [ ]:
def parse_remaining_lease_to_months(text):
    if pd.isna(text):
        return np.nan

    text = str(text).strip().lower()
    years = 0
    months = 0

    import re
    y = re.search(r"(\d+)\s+year", text)
    m = re.search(r"(\d+)\s+month", text)

    if y:
        years = int(y.group(1))
    if m:
        months = int(m.group(1))

    return years * 12 + months
    """Convert lease text such as years and months into a single numeric value measured in months."""

def parse_storey_mid(text):
    if pd.isna(text):
        return np.nan
    try:
        low, _, high = str(text).partition(" TO ")
        return (int(low) + int(high)) / 2
    except Exception:
        return np.nan
    """Approximate a floor-range string by its midpoint so storey information can be used numerically."""

model_df_full = df.copy()

# time features
model_df_full["month"] = pd.to_datetime(model_df_full["month"], format="%Y-%m")
model_df_full["sale_year"] = model_df_full["month"].dt.year
model_df_full["sale_month_num"] = model_df_full["month"].dt.month
model_df_full["sale_quarter"] = model_df_full["month"].dt.quarter

# lease / age features
model_df_full["remaining_lease_months"] = model_df_full["remaining_lease"].apply(parse_remaining_lease_to_months)
model_df_full["property_age_at_sale"] = model_df_full["sale_year"] - model_df_full["lease_commence_date"]
model_df_full["lease_balance_years"] = model_df_full["remaining_lease_months"] / 12.0

# floor / area related
model_df_full["storey_mid"] = model_df_full["storey_range"].apply(parse_storey_mid)
model_df_full["price_per_sqm"] = model_df_full["resale_price"] / model_df_full["floor_area_sqm"]

# simplified location key
model_df_full["address"] = model_df_full["block"].astype(str) + " " + model_df_full["street_name"].astype(str)

# log target for optional analysis if needed
model_df_full["log_resale_price"] = np.log1p(model_df_full["resale_price"])

model_df_full.head()

In [ ]:
engineered_numeric = [
    "floor_area_sqm",
    "lease_commence_date",
    "remaining_lease_months",
    "property_age_at_sale",
    "lease_balance_years",
    "storey_mid",
    "sale_year",
    "sale_month_num",
    "sale_quarter",
    "latitude",
    "longitude",
]

engineered_categorical = [
    "town",
    "flat_type",
    "flat_model",
    "street_name",
    # "storey_range",
]

keep_cols = engineered_numeric + engineered_categorical + ["resale_price", "price_per_sqm", "log_resale_price"]
model_df = model_df_full[keep_cols].copy()

print(model_df.shape)
model_df.head()


In [ ]:
corr1 = model_df.select_dtypes(include=(['int32', 'int64', 'float64'])).corr('spearman')['price_per_sqm'].drop(["price_per_sqm", "resale_price", "log_resale_price"]).sort_values(ascending=True)
corr2 = model_df.select_dtypes(include=(['int32', 'int64', 'float64'])).corr('spearman')['resale_price'].drop(["price_per_sqm", "resale_price", "log_resale_price"]).sort_values(ascending=True)

f, (ax1, ax2) =  plt.subplots(nrows=1, ncols=2, figsize=(12,5))

ax1.barh(corr1.index, corr1.values)
ax1.set_xlabel("Correlation with price_per_sqm")
ax1.set_title("Feature correlation with price_per_sqm")

ax2.barh(corr2.index, corr2.values)
ax2.set_xlabel("Correlation with resale_price")
ax2.set_title("Feature correlation with resale_price")

plt.show()

## 3. Train, Validate, Test Data

This section separates the dataset into different roles.

### Why use three splits?
- **training set**: fit model parameters
- **validation set**: compare candidate hyperparameters and model choices
- **test set**: provide a final out-of-sample evaluation

Keeping the test set untouched until the end reduces the risk of overfitting your modeling decisions to the evaluation data.


In [ ]:
X = model_df.drop(columns=["resale_price", "price_per_sqm", "log_resale_price"])
y = model_df["resale_price"]

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, random_state=RANDOM_STATE
)
# This gives about 70% train / 15% val / 15% test overall.

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


## 4. Helper functions and preprocessors

The next section defines reusable evaluation and preprocessing utilities.

### Why centralize this logic?
Without helper functions, each model section would repeat the same code for:
- metrics
- fitting
- validation
- random search over parameters
- storage of results

By abstracting those tasks into helper functions, the notebook becomes easier to read and easier to extend with new models later.


### Function rationale: common metrics and reusable evaluation helpers

The next code cell creates the reusable evaluation machinery for the notebook.

Important ideas here include:
- computing several error metrics instead of relying on just one
- storing fitted models and predictions in dictionaries for later comparison
- sampling rows for expensive models so tuning stays practical

These helpers keep the later model sections focused on modeling choices rather than bookkeeping.


In [ ]:
def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    nonzero_mask = y_true != 0
    mape = np.mean(np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask]) / y_true[nonzero_mask])) * 100

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "MAPE_pct": mape,
    }

results = []
fitted_models = {}
predictions = {}

def add_result(model_name, fitted_model, X_eval, y_eval, fit_seconds=None, notes=None):
    pred = fitted_model.predict(X_eval)
    metrics = regression_metrics(y_eval, pred)
    metrics["Model"] = model_name
    if fit_seconds is not None:
        metrics["Fit_seconds"] = fit_seconds
    if notes is not None:
        metrics["Notes"] = notes
    results.append(metrics)
    fitted_models[model_name] = fitted_model
    predictions[model_name] = pred
    return pred, metrics
    """Compute multiple regression metrics so model quality is judged from several complementary angles."""
    """Store a model's predictions and metrics in shared containers so later comparison and plotting are straightforward."""

def sample_rows(X_df, y_series, max_rows, random_state=RANDOM_STATE):
    if (max_rows is None) or (len(X_df) <= max_rows):
        return X_df.copy(), y_series.copy()
    sampled_index = X_df.sample(n=max_rows, random_state=random_state).index
    return X_df.loc[sampled_index].copy(), y_series.loc[sampled_index].copy()
    """Randomly subsample large datasets for tuning to keep slow models computationally manageable."""
    
def evaluate_on_validation(pipeline, X_tr, y_tr, X_va, y_va):
    pipeline.fit(X_tr, y_tr)
    pred = pipeline.predict(X_va)
    return regression_metrics(y_va, pred)

def tune_with_progress(
    model_name,
    pipeline_builder,
    param_distributions,
    X_tr,
    y_tr,
    X_va,
    y_va,
    n_iter=8,
    sample_limit=None,
    random_state=RANDOM_STATE
):
    X_tr_use, y_tr_use = X_tr, y_tr
    if sample_limit is not None:
        X_tr_use, y_tr_use = sample_rows(X_tr, y_tr, sample_limit, random_state=random_state)

    sampled_params = list(ParameterSampler(param_distributions, n_iter=n_iter, random_state=random_state))

    tuning_rows = []
    best_score = np.inf
    best_params = None
    best_pipeline = None

    print(f"Tuning {model_name} on {len(X_tr_use):,} training rows and {len(X_va):,} validation rows")

    for params in tqdm(sampled_params, desc=f"{model_name} tuning"):
        pipeline = pipeline_builder(**params)
        start = time.time()
        metrics = evaluate_on_validation(pipeline, X_tr_use, y_tr_use, X_va, y_va)
        elapsed = time.time() - start

        row = {"fit_seconds": elapsed, **params, **metrics}
        tuning_rows.append(row)

        if metrics["RMSE"] < best_score:
            best_score = metrics["RMSE"]
            best_params = params
            best_pipeline = pipeline

    tuning_df = pd.DataFrame(tuning_rows).sort_values("RMSE").reset_index(drop=True)
    return best_params, tuning_df, best_pipeline
    """Fit a pipeline on the training split and return validation metrics for one hyperparameter setting."""
    
def fit_model_on_sample(pipeline, X_train_data, y_train_data, sample_limit=None, desc=None):
    X_fit, y_fit = sample_rows(X_train_data, y_train_data, sample_limit, random_state=RANDOM_STATE)
    if desc is not None:
        print(f"{desc}: fitting on {len(X_fit):,} rows")
    start = time.time()
    pipeline.fit(X_fit, y_fit)
    fit_seconds = time.time() - start
    return pipeline, fit_seconds, len(X_fit)

### Preprocessing rationale: why there are two preprocessors

The notebook defines:

- a **general preprocessor** for models that benefit from scaled numeric inputs
- a **tree preprocessor** for tree-based models that do not need numeric scaling

This is a subtle but important machine-learning concept:  
different model families often need different preprocessing, and forcing them all through the same pipeline can hurt performance.


In [ ]:
# General preprocessing
numeric_features = engineered_numeric
categorical_features = engineered_categorical

general_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)

# Tree-friendly preprocessing: impute numerics, one-hot categoricals, no numeric scaling needed
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)


## 5. Linear Regression

Linear regression serves as the baseline model.

### Why start here?
It is simple, fast, and interpretable.  
Even if it is not the best-performing model, it shows how much predictive power is available from a mostly linear combination of features.

A good baseline is valuable because later models should justify their extra complexity with better performance.


In [ ]:
linear_model = Pipeline(steps=[
    ("preprocessor", general_preprocessor),
    ("model", LinearRegression())
])

start = time.time()
linear_model.fit(X_train_full, y_train_full)
linear_fit_seconds = time.time() - start

linear_pred, linear_metrics = add_result(
    "Linear Regression",
    linear_model,
    X_test,
    y_test,
    fit_seconds=linear_fit_seconds
)

linear_metrics

## 6. Polynomial Regression

Polynomial regression expands selected numeric relationships beyond straight lines.

### Why try this?
Housing prices are rarely perfectly linear in inputs such as floor area, location, or lease balance.  
Polynomial features let a linear model bend a bit while remaining easier to interpret than many fully non-linear models.


In [ ]:
poly_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)

poly_model = Pipeline(steps=[
    ("preprocessor", poly_preprocessor),
    ("model", LinearRegression())
])

start = time.time()
poly_model.fit(X_train_full, y_train_full)
poly_fit_seconds = time.time() - start

poly_pred, poly_metrics = add_result(
    "Polynomial Regression",
    poly_model,
    X_test,
    y_test,
    fit_seconds=poly_fit_seconds
)

poly_metrics

## 7. Random Forest

Random forests average many decision trees.

### Why they are useful here
They can capture non-linear effects and variable interactions without requiring heavy feature transformation.  
They are also fairly robust on mixed tabular data and provide feature importance measures for interpretation.


### Model-building rationale: Random Forest

The next function builds a full sklearn pipeline, not just a model object.

That means preprocessing and modeling are bundled together, which is good practice because:
- train / validation / test data are transformed consistently
- tuning is safer
- inference later uses exactly the same preprocessing steps as training


In [ ]:
def build_rf_pipeline(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
):
    return Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=True,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    """Construct a full preprocessing-plus-model pipeline for random forest regression."""

rf_param_distributions = {
    "n_estimators": [60, 100, 140],
    "max_depth": [14, 20, 28],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5],
}

rf_best_params, rf_tuning_df, _ = tune_with_progress(
    model_name="Random Forest",
    pipeline_builder=build_rf_pipeline,
    param_distributions=rf_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=6,
    sample_limit=30000
)

display(rf_tuning_df.head(10))

print("Best RF params:", rf_best_params)

best_rf_model = build_rf_pipeline(**rf_best_params)

RF_FINAL_SAMPLE = 60000
best_rf_model, rf_fit_seconds, rf_rows_used = fit_model_on_sample(
    best_rf_model,
    X_train_full,
    y_train_full,
    sample_limit=RF_FINAL_SAMPLE,
    desc="Random Forest"
)

rf_pred, rf_metrics = add_result(
    "Random Forest",
    best_rf_model,
    X_test,
    y_test,
    fit_seconds=rf_fit_seconds,
    notes=f"trained on {rf_rows_used:,} sampled rows"
)

rf_metrics

## 8. k-Nearest Neighbors

k-NN predicts a transaction by looking at similar past observations.

### Why include it?
It provides an intuitive benchmark:
"What would this flat likely sell for if we look at nearby points in feature space?"  

This can work surprisingly well on structured tabular problems, but it can also become sensitive to scaling and high dimensionality, which is why preprocessing matters a lot.


In [ ]:

def build_knn_pipeline(
    n_neighbors=20,
    weights="distance",
    p=2,
):
    return Pipeline(steps=[
        ("preprocessor", general_preprocessor),
        ("model", KNeighborsRegressor(
            n_neighbors=n_neighbors,
            weights=weights,
            p=p,
            n_jobs=-1
        ))
    ])
    """Construct a preprocessing-plus-model pipeline for k-nearest-neighbors regression."""
    
knn_param_distributions = {
    "n_neighbors": [5, 10, 20, 30, 40],
    "weights": ["uniform", "distance"],
    "p": [1, 2],
}

knn_best_params, knn_tuning_df, _ = tune_with_progress(
    model_name="KNN",
    pipeline_builder=build_knn_pipeline,
    param_distributions=knn_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=8,
    sample_limit=40000
)

display(knn_tuning_df.head(10))

best_knn_model = build_knn_pipeline(**knn_best_params)

start = time.time()
best_knn_model.fit(X_train_full, y_train_full)
knn_fit_seconds = time.time() - start

knn_pred, knn_metrics = add_result(
    "k-Nearest Neighbors",
    best_knn_model,
    X_test,
    y_test,
    fit_seconds=knn_fit_seconds
)

knn_metrics

## 9. Neural Network (MLPRegressor)

The multilayer perceptron is a simple feed-forward neural network for tabular regression.

### Why test it?
It gives the notebook a flexible non-linear model that is different from tree ensembles.  
It may capture patterns that linear regression misses, but it can also be harder to tune and more sensitive to preprocessing.


In [ ]:
def build_mlp_pipeline(
    hidden_layer_sizes=(64, 32),
    alpha=0.001,
    learning_rate_init=0.003,
):
    return Pipeline(steps=[
        ("preprocessor", general_preprocessor),
        ("model", MLPRegressor(
            hidden_layer_sizes=hidden_layer_sizes,
            alpha=alpha,
            learning_rate_init=learning_rate_init,
            max_iter=120,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=8,
            random_state=RANDOM_STATE
        ))
    ])
    """Construct a preprocessing-plus-model pipeline for a feed-forward neural network regressor."""
    
mlp_param_distributions = {
    "hidden_layer_sizes": [(32,), (64,), (64, 32)],
    "alpha": [1e-4, 1e-3, 1e-2],
    "learning_rate_init": [0.001, 0.003]
}

mlp_best_params, mlp_tuning_df, _ = tune_with_progress(
    model_name="MLP",
    pipeline_builder=build_mlp_pipeline,
    param_distributions=mlp_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=5,
    sample_limit=25000
)

display(mlp_tuning_df.head(10))

print("Best MLP params:", mlp_best_params)

best_mlp_model = build_mlp_pipeline(**mlp_best_params)

MLP_FINAL_SAMPLE = 40000
best_mlp_model, mlp_fit_seconds, mlp_rows_used = fit_model_on_sample(
    best_mlp_model,
    X_train_full,
    y_train_full,
    sample_limit=MLP_FINAL_SAMPLE,
    desc="MLP"
)

mlp_pred, mlp_metrics = add_result(
    "Neural Network (MLP)",
    best_mlp_model,
    X_test,
    y_test,
    fit_seconds=mlp_fit_seconds,
    notes=f"trained on {mlp_rows_used:,} sampled rows"
)

mlp_metrics

## 10. XGBoost

XGBoost is a gradient-boosted tree model.

### Why it is often strong on tabular data
Boosting builds trees sequentially, with each new tree correcting errors made by the previous ones.  
That often leads to high predictive accuracy on structured datasets like housing transactions.


### Model-building rationale: boosted trees

The next two boosted-tree sections explore models that often perform very well on tabular data.

The tuning grids are intentionally modest rather than huge.  
That is a practical compromise: enough variation to compare sensible hyperparameters, but not so much that the notebook becomes painfully slow to run.


In [ ]:
def build_xgb_pipeline(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
):
    return Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", XGBRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    """Construct a preprocessing-plus-model pipeline for XGBoost regression."""
    
xgb_param_distributions = {
    "n_estimators": [200, 300, 500],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

xgb_best_params, xgb_tuning_df, _ = tune_with_progress(
    model_name="XGBoost",
    pipeline_builder=build_xgb_pipeline,
    param_distributions=xgb_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=8,
    sample_limit=80000
)

display(xgb_tuning_df.head(10))

best_xgb_model = build_xgb_pipeline(**xgb_best_params)

start = time.time()
best_xgb_model.fit(X_train_full, y_train_full)
xgb_fit_seconds = time.time() - start

xgb_pred, xgb_metrics = add_result(
    "XGBoost",
    best_xgb_model,
    X_test,
    y_test,
    fit_seconds=xgb_fit_seconds
)

display(xgb_metrics)

## 11. LightGBM

LightGBM is another gradient-boosted tree framework optimized for speed and scalability.

### Why compare it with XGBoost?
Both are strong boosted-tree methods, but they differ in implementation details and sometimes in accuracy / training speed trade-offs.  
Testing both helps show whether the dataset clearly favors one boosting strategy over the other.


In [ ]:
def build_lgbm_pipeline(
    n_estimators=400,
    num_leaves=31,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
):
    return Pipeline(steps=[
        ("preprocessor", tree_preprocessor),
        ("model", LGBMRegressor(
            n_estimators=n_estimators,
            num_leaves=num_leaves,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    """Construct a preprocessing-plus-model pipeline for LightGBM regression."""

lgbm_param_distributions = {
    "n_estimators": [250, 400, 600],
    "num_leaves": [31, 63, 127],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
}

lgbm_best_params, lgbm_tuning_df, _ = tune_with_progress(
    model_name="LightGBM",
    pipeline_builder=build_lgbm_pipeline,
    param_distributions=lgbm_param_distributions,
    X_tr=X_train,
    y_tr=y_train,
    X_va=X_val,
    y_va=y_val,
    n_iter=8,
    sample_limit=80000
)

display(lgbm_tuning_df.head(10))

best_lgbm_model = build_lgbm_pipeline(**lgbm_best_params)

start = time.time()
best_lgbm_model.fit(X_train_full, y_train_full)
lgbm_fit_seconds = time.time() - start

lgbm_pred, lgbm_metrics = add_result(
    "LightGBM",
    best_lgbm_model,
    X_test,
    y_test,
    fit_seconds=lgbm_fit_seconds
)

display(lgbm_metrics)

## 12. Model comparison

Once all candidate models are fitted, this section puts their metrics side by side.

### What to look for
- lower RMSE / MAE means smaller prediction errors
- higher R² means the model explains more variance
- lower MAPE means smaller percentage error on average

This is where model selection becomes evidence-based rather than intuition-based.


In [ ]:
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    ordered_cols = ["Model", "RMSE", "MAE", "R2", "MAPE_pct", "Fit_seconds", "Notes"]
    existing_cols = [col for col in ordered_cols if col in results_df.columns]
    results_df = results_df[existing_cols].sort_values("RMSE").reset_index(drop=True)

display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print("Best model based on lowest test RMSE:", best_model_name)

In [ ]:

plt.figure(figsize=(6, 3))
plt.bar(results_df["Model"], results_df["RMSE"])
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison by RMSE")
plt.ylabel("RMSE")
plt.show()

plt.figure(figsize=(6, 3))
plt.bar(results_df["Model"], results_df["R2"])
plt.xticks(rotation=30, ha="right")
plt.title("Model Comparison by R²")
plt.ylabel("R²")
plt.show()


## 13. Actual vs predicted plots

Summary metrics are useful, but plots reveal error structure that tables can miss.

### Why these plots matter
They help you see whether a model:
- systematically underpredicts expensive flats
- overpredicts cheaper flats
- produces too much variance
- tracks the diagonal well across the price range

A model can have reasonable aggregate metrics yet still behave poorly in important parts of the target distribution.


In [ ]:
f, ([ax1, ax2, ax3, ax4], [ax5, ax6, ax7, ax8]) =  plt.subplots(2, 4, figsize=(24,12))

for i in range(0,len(results_df["Model"])):
    model_name = results_df["Model"][i]
    axes =[ax1, ax2, ax3, ax4, ax5, ax6, ax7, ax8][i]
    
    pred = predictions[model_name]
    axes.scatter(y_test, pred, alpha=0.25, s=10)
    min_val = min(y_test.min(), pred.min())
    max_val = max(y_test.max(), pred.max())
    axes.plot([min_val, max_val], [min_val, max_val], linestyle="--")
    axes.set_title(f"Actual vs Predicted: {model_name}")
    axes.set_xlabel("Actual Price")
    axes.set_ylabel("Predicted Price")

plt.show()


## 14. Feature importance and SHAP

After choosing strong tree-based models, this section asks a different question:

### Not just "which model wins?"
But also:
- which features matter most?
- how do they push predictions upward or downward?

Feature importance gives a quick global ranking.  
SHAP adds a more local explanation framework so you can see how features contribute to individual predictions.


In [ ]:
tree_model_candidates = [name for name in results_df["Model"] if name in ["LightGBM", "XGBoost", "Random Forest"]]
best_tree_model_name = tree_model_candidates[0] if len(tree_model_candidates) > 0 else None
print("Best tree-based model:", best_tree_model_name)

In [ ]:
best_tree_model = fitted_models[best_tree_model_name]
tree_pre = best_tree_model.named_steps["preprocessor"]
tree_est = best_tree_model.named_steps["model"]

feature_names = tree_pre.get_feature_names_out()

if hasattr(tree_est, "feature_importances_"):
    feature_importances = pd.DataFrame({
        "feature": feature_names,
        "importance": tree_est.feature_importances_
    }).sort_values("importance", ascending=False)

    display(feature_importances.head(20))

    top_n = 20
    top_feat = feature_importances.head(top_n).sort_values("importance")
    plt.figure(figsize=(8, 4))
    plt.barh(top_feat["feature"], top_feat["importance"])
    plt.title(f"Top {top_n} Feature Importances: {best_tree_model_name}")
    plt.xlabel("Importance")
    plt.show()


In [ ]:
best_tree_model = fitted_models[best_tree_model_name]
tree_pre = best_tree_model.named_steps["preprocessor"]
tree_est = best_tree_model.named_steps["model"]

X_shap_sample, y_shap_sample = sample_rows(X_test, y_test, max_rows=1200, random_state=RANDOM_STATE)
X_shap_transformed = tree_pre.transform(X_shap_sample)

if hasattr(X_shap_transformed, "toarray"):
    # Keep sample moderate to avoid excessive memory use
    X_shap_dense = X_shap_transformed.toarray()
else:
    X_shap_dense = X_shap_transformed

try:
    explainer = shap.Explainer(tree_est, X_shap_dense, feature_names=tree_pre.get_feature_names_out())
    shap_values = explainer(X_shap_dense)

    shap.plots.beeswarm(shap_values, max_display=20)
except Exception as e:
    print("SHAP calculation failed for this model/environment.")
    print("Details:", e)


## 15. Singapore geospatial maps and market heatmaps

Even in a prediction notebook, it helps to step back into exploratory visualization.

### Why include maps and heatmaps here?
They connect the modeling results back to domain intuition:
- where prices cluster spatially
- how towns differ over time
- how flat types vary across locations
- where transaction volume is concentrated

These views help the learner see the structure the models are trying to capture.


### Mapping rationale inside the prediction notebook

The next functions revisit geospatial plotting, but now for explanatory purposes inside the modeling workflow.

This is useful because a prediction notebook should not become a black box.  
Looking at maps and heatmaps helps connect model performance back to the real spatial structure of the housing market.


In [ ]:
def percentile_clip(series: pd.Series, low: float = 0.1, high: float = 0.9) -> tuple[float, float]:
    if isinstance(series, pd.DataFrame):
        series = series.iloc[:, 0]
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return (np.nan, np.nan)
    return (float(s.quantile(low)), float(s.quantile(high)))
    """Return quantile bounds used to clip mapped values so outliers do not dominate the color scale."""

def plot_metric_map(
    gdf: gpd.GeoDataFrame,
    metric_col: str,
    title: str,
    # output_pdf: Path,
    cmap: str = "viridis",
    clip_quantiles: tuple[float, float] = (0.02, 0.98),
    center_zero: bool = False,
    marker_size: int = 22,
    legend_label: str | None = None,
):
    data = gdf.dropna(subset=[metric_col]).copy()
    if data.empty:
        raise ValueError(f"No data available for {metric_col}")

    q_low, q_high = percentile_clip(data[metric_col], *clip_quantiles)
    data["metric_clipped"] = data[metric_col].clip(q_low, q_high)

    data_3857 = data.to_crs(epsg=3857)

    fig, ax = plt.subplots(figsize=(9, 10))

    if center_zero:
        bound = max(abs(q_low), abs(q_high))
        norm = TwoSlopeNorm(vmin=-bound, vcenter=0.0, vmax=bound)
    else:
        norm = Normalize(vmin=q_low, vmax=q_high)

    data_3857.plot(
        ax=ax,
        column="metric_clipped",
        cmap=cmap,
        markersize=marker_size,
        alpha=0.9,
        legend=True,
        norm=norm,
        legend_kwds={"shrink": 0.35, "label": legend_label or metric_col},
        zorder=3,
    )
    
    cx.add_basemap(ax, source=cx.providers.CartoDB.Positron, attribution=False)

    ax.set_title(title, fontsize=14)
    ax.set_axis_off()

    # Zoom to the buildings with a small margin
    xmin, ymin, xmax, ymax = data_3857.total_bounds
    xpad = (xmax - xmin) * 0.05
    ypad = (ymax - ymin) * 0.05
    ax.set_xlim(xmin - xpad, xmax + xpad)
    ax.set_ylim(ymin - ypad, ymax + ypad)

    plt.tight_layout()
    # plt.savefig(output_pdf, bbox_inches="tight")
    plt.show()
    """Plot a geospatial point map for a chosen metric using clipped values to improve visual readability."""


In [ ]:
def build_geodf(df: pd.DataFrame) -> gpd.GeoDataFrame:
    out = df.copy()
    geometry = [Point(xy) for xy in zip(out["longitude"], out["latitude"])]
    gdf = gpd.GeoDataFrame(out, geometry=geometry, crs="EPSG:4326")
    return gdf
    """Convert tabular latitude and longitude columns into a GeoDataFrame for mapping."""

building_metrics_year = (
    model_df_full
    .groupby(["sale_year", "address"], as_index=False)
    .agg(
        median_psm=("price_per_sqm", "median"),
        txn_count=("price_per_sqm", "size"),
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        address=("address", "first"),
        town=("town", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
    )
)

building_metrics = (
    model_df_full
    .groupby(["address"], as_index=False)
    .agg(
        median_psm=("price_per_sqm", "median"),
        txn_count=("price_per_sqm", "size"),
        latitude=("latitude", "median"),
        longitude=("longitude", "median"),
        address=("address", "first"),
        town=("town", lambda s: s.mode().iloc[0] if not s.mode().empty else s.iloc[0]),
    )
)

gdf_buildings_year = build_geodf(building_metrics_year)
gdf_buildings = build_geodf(building_metrics)


In [ ]:
plot_metric_map(
    gdf_buildings_year[gdf_buildings_year.sale_year==2025],
    metric_col="median_psm",
    title="Singapore HDB resale market — \n median price per sqm by building (Year 2025)",
    cmap="RdYlBu_r",
    clip_quantiles=(0.10, 0.90),
    center_zero=False,
    marker_size=1,
    legend_label="SGD per square meter",
)

In [ ]:
plot_metric_map(
    gdf_buildings,
    metric_col="median_psm",
    title="Singapore HDB resale market — \n median price per sqm by building (2017-2026)",
    cmap="RdYlBu_r",
    clip_quantiles=(0.10, 0.90),
    center_zero=False,
    marker_size=1,
    legend_label="SGD per square meter",
)

In [ ]:
plot_metric_map(
    gdf_buildings,
    metric_col="txn_count",
    title="Singapore HDB resale market — \n transaction counts by building (2017-2026)",
    clip_quantiles=(0.10, 0.90),
    center_zero=False,
    marker_size=1,
    legend_label="SGD per square meter",
)


In [ ]:
# Median resale price by town and sale year
heatmap_year = (
    df.assign(sale_year=pd.to_datetime(df["month"]).dt.year)
      .groupby(["town", "sale_year"])["resale_price"]
      .median()
      .unstack()
)

heatmap_year = heatmap_year.loc[heatmap_year.mean(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(7, 5))
plt.imshow(heatmap_year, aspect="auto", cmap='RdYlBu_r')
plt.colorbar(label="Median Resale Price")
plt.yticks(range(len(heatmap_year.index)), heatmap_year.index)
plt.xticks(range(len(heatmap_year.columns)), heatmap_year.columns, rotation=45)
plt.title("Median HDB Resale Price by Town and Year")
plt.xlabel("Year")
plt.ylabel("Town")
plt.tight_layout()
plt.show()


In [ ]:

# Median resale price by town and flat type
heatmap_flat = (
    df.groupby(["town", "flat_type"])["resale_price"]
      .median()
      .unstack()
)

heatmap_flat = heatmap_flat.loc[heatmap_flat.mean(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(7, 5))
plt.imshow(heatmap_flat, aspect="auto", cmap="RdYlBu_r")
plt.colorbar(label="Median Resale Price")
plt.yticks(range(len(heatmap_flat.index)), heatmap_flat.index)
plt.xticks(range(len(heatmap_flat.columns)), heatmap_flat.columns, rotation=45)
plt.title("Median HDB Resale Price by Town and Flat Type")
plt.xlabel("Flat Type")
plt.ylabel("Town")
plt.tight_layout()
plt.show()

In [ ]:
# Transaction volume by town and year
volume_heatmap = (
    df.assign(sale_year=pd.to_datetime(df["month"]).dt.year)
      .groupby(["town", "sale_year"])
      .size()
      .unstack(fill_value=0)
)

volume_heatmap = volume_heatmap.loc[volume_heatmap.sum(axis=1).sort_values(ascending=False).index]

plt.figure(figsize=(7, 5))
plt.imshow(volume_heatmap, aspect="auto")
plt.colorbar(label="Transaction Count")
plt.yticks(range(len(volume_heatmap.index)), volume_heatmap.index)
plt.xticks(range(len(volume_heatmap.columns)), volume_heatmap.columns, rotation=45)
plt.title("HDB Resale Transaction Volume by Town and Year")
plt.xlabel("Year")
plt.ylabel("Town")
plt.tight_layout()
plt.show()


### Why dimension reduction is exploratory rather than predictive

PCA, t-SNE, and UMAP are powerful visualization tools, but their 2D layouts should not be over-interpreted as literal geography or causal structure.

They are best viewed as:
- a way to inspect patterns
- a way to spot clusters or gradients
- a way to build intuition about the feature space


## 16. Dimension reduction for visualization

The feature space can be high-dimensional after one-hot encoding and numeric transformations.

### Why use PCA, t-SNE, and UMAP?
These methods project many dimensions down to two so we can visually inspect structure such as:
- clustering by town
- gradients in price
- separation between common and unusual transactions

This is not used directly for prediction; it is mainly an intuition-building tool.


In [ ]:

sample_n = min(3000, len(model_df))
viz_df = model_df.sample(sample_n, random_state=RANDOM_STATE).copy()

X_viz = viz_df.drop(columns=["resale_price", "price_per_sqm", "log_resale_price"])
y_viz = viz_df["resale_price"]

viz_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20))
        ]), categorical_features),
    ]
)

X_viz_transformed = viz_preprocessor.fit_transform(X_viz)

if hasattr(X_viz_transformed, "toarray"):
    X_viz_dense = X_viz_transformed.toarray()
else:
    X_viz_dense = X_viz_transformed

print("Visualization sample size:", len(viz_df))
print("Transformed visualization matrix shape:", X_viz_dense.shape)


In [ ]:

# PCA
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_viz_dense)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_viz, s=12)
plt.colorbar(sc, label="Resale Price")
plt.title("PCA of HDB Resale Data")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)


In [ ]:

# t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=30,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE
)
X_tsne = tsne.fit_transform(X_viz_dense)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_viz, s=12)
plt.colorbar(sc, label="Resale Price")
plt.title("t-SNE of HDB Resale Data")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.show()


In [ ]:
# UMAP
reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
X_umap = reducer.fit_transform(X_viz_dense)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_umap[:, 0], X_umap[:, 1], c=y_viz, s=12)
plt.colorbar(sc, label="Resale Price")
plt.title("UMAP of HDB Resale Data")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.show()

## 17. Final summary

The notebook ends by summarizing the model comparison and reminding the reader how to interpret the results.

### Big picture
This notebook is meant to teach a full applied workflow:
- clean data carefully
- engineer meaningful features
- compare several model families fairly
- evaluate on held-out data
- interpret the winning model rather than stopping at a score


In [ ]:
display(results_df)

print("Best model:", best_model_name)
print()
print("Interpretation tips:")
print("- Lower RMSE / MAE is better.")
print("- Higher R² is better.")
print("- Random Forest and MLP in this notebook are intentionally trained on sampled rows to keep runtime practical.")
print("- If you want the absolute best accuracy later, you can increase RF_FINAL_SAMPLE and MLP_FINAL_SAMPLE gradually.")
print("- The geospatial maps use the geocoded latitude / longitude columns you added.")
print("- Tree boosting models often outperform plain linear models on housing data if the package is available.")
print("- KNN can also be slow when predicting on very large sets.")